# Project: Multilingual Search and Recsys over Wikipedia

https://www.nlplanet.org/course-practical-nlp/02-practical-nlp-first-tasks/11-multilingual-search-recsys-wikipedia

In [1]:
from datasets import load_dataset
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss

Download and prepare dataset

In [3]:
# download part of the Italian Wikipedia dataset
dataset = load_dataset("wikipedia", "20220301.it", split="train",trust_remote_code=True)
print(dataset)

train-00000-of-00010.parquet:   0%|          | 0.00/487M [00:00<?, ?B/s]

train-00001-of-00010.parquet:   0%|          | 0.00/355M [00:00<?, ?B/s]

train-00002-of-00010.parquet:   0%|          | 0.00/313M [00:00<?, ?B/s]

train-00003-of-00010.parquet:   0%|          | 0.00/282M [00:00<?, ?B/s]

train-00004-of-00010.parquet:   0%|          | 0.00/225M [00:00<?, ?B/s]

...

train-00007-of-00010.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

train-00008-of-00010.parquet:   0%|          | 0.00/198M [00:00<?, ?B/s]

train-00009-of-00010.parquet:   0%|          | 0.00/222M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1743035 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 1743035
})


In [4]:
# keep only first 10k articles to make computations faster
dataset_subset = dataset.train_test_split(train_size=10000)["train"]
print(dataset_subset)

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 10000
})


In [5]:
df = pd.DataFrame(dataset_subset)
df.head()

,id,url,title,text
0,4155314,https://it.wikipedia.org/wiki/Eurydema%20ventr...,Eurydema ventralis,La cimice dei cavolfiori (Eurydema ventralis )...
1,30856,https://it.wikipedia.org/wiki/Mendatica,Mendatica,Mendatica (Mendàiga o Mendéga in ligure) è un ...
2,9198252,https://it.wikipedia.org/wiki/Saski%20Baskonia...,Saski Baskonia 2005-2006,Questa voce raccoglie le informazioni riguarda...
3,1083996,https://it.wikipedia.org/wiki/Paolino%20Paperi...,Paolino Paperino Band,La Paolino Paperino Band è un gruppo punk rock...
4,156998,https://it.wikipedia.org/wiki/Molinchart,Molinchart,Molinchart è un comune francese di 325 abitant...


In [6]:
# join article title and text into a single column
df["full_text"] = df["title"] + ". " + df["text"]

Create article embeddings

In [7]:
# download the sentence embeddings model
embedder = SentenceTransformer('distiluse-base-multilingual-cased-v1')

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

c:\Users\TristramArmour\anaconda3\envs\learning\Lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\TristramArmour\.cache\huggingface\hub\models--sentence-transformers--distiluse-base-multilingual-cased-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

2_Dense/config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

In [8]:
# embed article texts
corpus_embeddings = embedder.encode(df["full_text"].values)
print(corpus_embeddings.shape)

(10000, 512)


In [9]:
corpus_embeddings

array([[ 0.01199902,  0.01860929,  0.03671446, ..., -0.0611642 ,
         0.0208418 ,  0.0365338 ],
       [ 0.0470414 , -0.0776713 , -0.02880744, ..., -0.03608015,
        -0.02907495, -0.04080971],
       [ 0.07091886,  0.06604121,  0.02907223, ...,  0.03134548,
         0.08870167,  0.05852308],
       ...,
       [-0.02026301, -0.09221597,  0.02084875, ..., -0.00053824,
         0.000352  ,  0.00634732],
       [-0.01826463, -0.01165413, -0.03922078, ...,  0.0482648 ,
         0.03307712,  0.01952555],
       [ 0.08688423, -0.04505253, -0.06366456, ..., -0.01472316,
        -0.023554  , -0.00638078]], dtype=float32)

Create faiss index

In [10]:
# create faiss index
n_cells = 1000
num_dimensions = corpus_embeddings.shape[1]
quantizer = faiss.IndexFlatL2(num_dimensions)
index = faiss.IndexIVFFlat(quantizer, num_dimensions, n_cells)
index.train(corpus_embeddings)
index.add(corpus_embeddings)

Semantic search

In [11]:
# search
text_query = "seconda guerra mondiale"
query_embedding = embedder.encode([text_query])
num_results = 5
distances, indexes = index.search(query_embedding, num_results)

In [12]:
# show results
relevant_rows = df.iloc[indexes[0]]
for i,row in relevant_rows.iterrows():
  print(f"- {row['title']}")

- Ordine di battaglia della seconda battaglia di El Alamein
- Sfollamento di Viareggio
- Battaglia dello Stretto della Sonda
- Guerra dei canederli di Pasing
- Bulgaria nella seconda guerra mondiale


Recommender system

In [13]:
# choose an article
article_row = df.iloc[10] # Salame di Cioccolato
print(article_row["title"])

Dimitrije Ljotić


In [15]:
text_query = article_row["text"]
query_embedding = embedder.encode([text_query])
num_results = 5
distances, indexes = index.search(query_embedding, num_results)

In [17]:
# show results
relevant_rows = df.iloc[indexes[0]]
for i,row in relevant_rows.iterrows():
  print(f"- {row['title']} - {row['text']}")

- Dimitrije Ljotić - 

Biografia 
Suo padre, Vladimir Ljotić fu membro del parlamento serbo, console in Tessalonica e sindaco di Smederevo. L'infanzia di Dimitrije seguì quindi la carriera paterna: nacque a Belgrado, iniziò le scuole a Smederevo, si diplomò a 16 anni al liceo serbo in Tessalonica prima di laurearsi in giurisprudenza a Belgrado.

Durante le guerre balcaniche prestò servizio da volontario come medico. Nel 1913 si trasferì a Parigi, dove rimase sino all'inizio della prima guerra mondiale quando ritornò in Serbia e si arruolò nell'esercito. Dopo la guerra prestò servizio come comandante in una stazione ferroviaria a Buccari in Croazia, dove si fece notare per aver interrotto uno sciopero arrestando 36 lavoratori. A Buccari conosce la donna che diverrà sua moglie, Ivka, con cui tornò a Smederevo nel 1920 e iniziò a esercitare con la sua laurea in legge. A Smederevo si unì al partito radicale diventando presto presidente della sezione giovanile. Nel 1931 il re Alessandro I l